# VR AI Denoiser: CUDA runs on Colab / Kaggle

Runs the parts of the project that need an NVIDIA GPU: the CUDA build, Phase 2
(CPU vs GPU), the full dataset, training, and the Phase 8/9 benchmarks.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

**Each new session:** run the cells in section 0 again (setup and build). Then
continue from the phase you stopped at. The dataset, checkpoints and results are
stored on Google Drive, so a disconnect loses only the phase that was running.

## 0. Setup (run every session)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2
!cmake --version | head -1
import torch; print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU - set Runtime > Change runtime type > T4 GPU"

In [ ]:
# Persistent storage on Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/vr-ai-denoiser-runs'
REPO = '/content/vr-ai-denoiser'
import os; os.makedirs(DRIVE, exist_ok=True)

**Getting the code.** Use one of these:
- **A (private GitHub repo):** create a *fine-grained* token at github.com → Settings → Developer settings,
  with read-only *Contents* access to `vr-ai-denoiser`. Add it in Colab's 🔑 *Secrets* panel as `GITHUB_TOKEN`,
  with notebook access switched on.
- **B (no GitHub):** zip the project on your Mac (leave out `build*`, `.venv`, `dataset/raw`, `outputs`),
  upload it to `MyDrive/vr-ai-denoiser-runs/vr-ai-denoiser.zip`, and leave the token secret unset.

In [ ]:
import os, subprocess, shutil
if not os.path.exists(REPO):
    zip_path = f'{DRIVE}/vr-ai-denoiser.zip'
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    if token:
        subprocess.run(['git', 'clone', '--depth', '1',
                        f'https://{token}@github.com/nidhivk09/vr-ai-denoiser.git', REPO], check=True)
    elif os.path.exists(zip_path):
        subprocess.run(['unzip', '-q', zip_path, '-d', '/content/unzip'], check=True)
        inner = [d for d in os.listdir('/content/unzip') if not d.startswith('__')]
        src = f'/content/unzip/{inner[0]}' if len(inner) == 1 else '/content/unzip'
        shutil.move(src, REPO)
    else:
        raise SystemExit('No GITHUB_TOKEN secret and no zip on Drive - see the cell above')
%cd {REPO}
!git log --oneline -1 2>/dev/null || ls

In [ ]:
# Point the large generated folders at Drive so they survive disconnects.
import os, shutil
links = {'dataset/raw': 'dataset_raw', 'dataset/patches': 'dataset_patches', 'dataset/scenes': 'dataset_scenes',
         'denoiser/checkpoints': 'checkpoints', 'outputs': 'outputs', 'benchmarks/results': 'results'}
for local, remote in links.items():
    dst = f'{DRIVE}/{remote}'
    os.makedirs(dst, exist_ok=True)
    path = f'{REPO}/{local}'
    if os.path.islink(path):
        continue
    if os.path.isdir(path):
        shutil.copytree(path, dst, dirs_exist_ok=True)
        shutil.rmtree(path)
    os.symlink(dst, path)
    print(f'{local:22s} -> {dst}')

In [ ]:
!pip install -q -r requirements.txt
# Build: CUDA renderer + LibTorch pipeline (LibTorch comes from the pip torch install). T4 = sm_75.
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=75 \
      -DCMAKE_PREFIX_PATH="$(python -c 'import torch; print(torch.utils.cmake_prefix_path)')" \
      > build_configure.log 2>&1; tail -5 build_configure.log
!cmake --build build -j4 > build.log 2>&1; echo "build exit code: $?"; grep -E "error|warning: |Built target" build.log | head -40
!cp build.log build_configure.log {DRIVE}/

If the build fails, send back `build.log` (it is also copied to Drive). This is the first time the CUDA code is compiled, so errors here are expected and fixable.

In [ ]:
!ctest --test-dir build --output-on-failure | tail -3
!python -m unittest discover -s tests 2>&1 | tail -3
!./build/renderer/pathtracer --info

## Phase 2 - CPU vs GPU agreement and speed-up, block-size sweep

In [ ]:
!python scripts/run_phase.py 2 2>&1 | tee {DRIVE}/phase2.log | tail -40

## Phases 3-4 - feature buffers and stereo

In [ ]:
!python scripts/run_phase.py 3 4 2>&1 | tee {DRIVE}/phase34.log | tail -15
from IPython.display import Image, display
display(Image('outputs/phase4/anaglyph.png', width=400))

## Phase 5 - full dataset
24 scenes × (1, 4, 8 spp + 2048 spp ground truth) × 2 eyes at 512². The cell skips
itself if the patches are already on Drive.

In [ ]:
import os
if os.path.exists('dataset/patches/stats.json'):
    print('dataset already on Drive - skipping (delete MyDrive/vr-ai-denoiser-runs/dataset_* to regenerate)')
else:
    !python scripts/run_phase.py 5 2>&1 | tee {DRIVE}/phase5.log | tail -30

## Phases 6-7 - training
Free Colab sessions last a few hours at most and disconnect when idle, so keep this tab open.
`EPOCHS = 60` is shorter than the default 150 so each run fits in a session.
The λ sweep resumes across sessions: a λ with a finished checkpoint is skipped.
**Kaggle is better for long runs** (30 GPU-hours/week, 12 h sessions; see the note at the end).

In [ ]:
EPOCHS = 60
LAMBDAS = '0,0.5,1,2'

# A run that was cut off still has a best.pt, which would make the sweep skip it.
# Remove any run that did not reach its planned epoch count.
import json, csv, shutil, pathlib
for run in pathlib.Path('denoiser/checkpoints').glob('*/'):
    a, log = run / 'args.json', run / 'log.csv'
    if not a.exists() or not log.exists():
        continue
    args = json.loads(a.read_text())
    done = len(list(csv.DictReader(open(log))))
    if done < args['epochs'] and not args.get('patience'):
        print(f'removing unfinished run {run.name} ({done}/{args["epochs"]} epochs)')
        shutil.rmtree(run)

In [ ]:
import os
if os.path.exists('denoiser/checkpoints/baseline/best.pt'):
    print('baseline already trained - skipping (delete its folder on Drive to retrain)')
else:
    !python scripts/run_phase.py 6 --epochs {EPOCHS} 2>&1 | tee {DRIVE}/phase6.log | grep -vE "^epoch +[0-9]+ " | tail -30

In [ ]:
!python scripts/run_phase.py 7 --epochs {EPOCHS} --lambdas {LAMBDAS} 2>&1 | tee -a {DRIVE}/phase7.log | grep -vE "^epoch +[0-9]+ " | tail -40

## Phases 8-9 - TorchScript + C++ pipeline, benchmarks (T4: 320 GB/s)

In [ ]:
!python scripts/run_phase.py 8 2>&1 | tee {DRIVE}/phase8.log | tail -30

In [ ]:
# Nsight Systems: Colab may not include nsys. This tries to find or install it; if that fails, Phase 9 skips Nsight and continues.
import shutil, glob, os
if not shutil.which('nsys'):
    found = glob.glob('/usr/local/cuda*/bin/nsys') + glob.glob('/opt/nvidia/nsight-systems/*/bin/nsys')
    if not found:
        !apt-get -qq install -y nsight-systems > /dev/null 2>&1 || apt-get -qq install -y cuda-nsight-systems-12-* > /dev/null 2>&1
        found = glob.glob('/opt/nvidia/nsight-systems/*/bin/nsys') + glob.glob('/usr/local/cuda*/bin/nsys')
    if found:
        os.environ['PATH'] = os.path.dirname(found[0]) + ':' + os.environ['PATH']
print('nsys:', shutil.which('nsys') or 'not available (Phase 9 will skip Nsight)')

In [ ]:
!python scripts/run_phase.py 9 --gpu-bandwidth 320 2>&1 | tee {DRIVE}/phase9.log | tail -40

## Collect results to send back

In [ ]:
!cd {REPO} && zip -qr {DRIVE}/results_bundle.zip benchmarks/results -x 'benchmarks/results/frames/*' && ls -lh {DRIVE}/results_bundle.zip

Everything is in **MyDrive/vr-ai-denoiser-runs/**: `results_bundle.zip`, the
`phase*.log` files and `build.log`.

### Running on Kaggle instead
Create a Notebook, set *Accelerator → GPU T4 x2* and switch *Internet* on. Then:
skip the Drive cells, set `DRIVE = '/kaggle/working/runs'` and `REPO = '/kaggle/working/vr-ai-denoiser'`,
get the code with `git clone` (add the token through *Add-ons → Secrets*, read with
`from kaggle_secrets import UserSecretsClient`) or attach the zip as a Dataset.
`/kaggle/working` is kept when you *Save Version* (with "Save & Run All").